# Error Analysis

Explore model errors interactively after running `scripts/evaluate_model.py`.

> Load the saved error analysis CSV from `artifacts/` and visualise confusion patterns.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.evaluation.error_analysis import analyze_errors
from src.evaluation.confusion import plot_confusion
from src.utils.paths import PATHS

## 1. Load error report (or synthesise one for demonstration)

In [ ]:
error_csv = PATHS.artifacts_dir / 'error_analysis.csv'

if error_csv.exists():
    df = pd.read_csv(error_csv)
    print(f'Loaded {len(df)} misclassified samples from {error_csv}')
else:
    # Synthetic demonstration
    rng = np.random.default_rng(0)
    labels = list('ABCDEFGHIJ')
    n = 200
    y_true = rng.choice(labels, n)
    y_pred = y_true.copy()
    # Inject ~20 % errors
    err_idx = rng.choice(n, size=int(n * 0.2), replace=False)
    y_pred[err_idx] = rng.choice(labels, size=len(err_idx))

    df = analyze_errors(y_true.tolist(), y_pred.tolist())
    print(f'Synthesised {len(df)} errors for demonstration')

df.head()

## 2. Most confused class pairs

In [ ]:
pairs = (
    df.groupby(['true', 'predicted'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)
print('Top confused pairs:')
print(pairs.head(10).to_string(index=False))

## 3. Error distribution by true class

In [ ]:
err_by_class = df['true'].value_counts()

plt.figure(figsize=(10, 4))
err_by_class.plot(kind='bar')
plt.title('Number of errors per true class')
plt.xlabel('True class')
plt.ylabel('Error count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Confusion matrix (subset)

In [ ]:
# Use the full y_true / y_pred if available, otherwise rebuild from the errors DataFrame
labels_used = sorted(set(df['true'].tolist()) | set(df['predicted'].tolist()))

# For a meaningful matrix we need all predictions, not just errors.
# Patch: treat non-error samples as correct (true == predicted).
all_true = df['true'].tolist()
all_pred = df['predicted'].tolist()

out_path = str(PATHS.artifacts_dir / 'confusion_notebook.png')
matrix = plot_confusion(all_true, all_pred, labels=labels_used, out_path=out_path)
print(f'Confusion matrix saved to {out_path}')

## 5. Error-rate summary table

In [ ]:
n_errors = len(df)
# If we only have errors, error rate is approximate
print(f'Total misclassified samples in report: {n_errors}')

error_rate_per_pair = (
    pairs
    .assign(pair=pairs['true'] + '→' + pairs['predicted'])
    .set_index('pair')['count']
    / n_errors
).head(15)

plt.figure(figsize=(10, 4))
error_rate_per_pair.plot(kind='barh')
plt.title('Relative share of each confusion pair')
plt.xlabel('Fraction of errors')
plt.tight_layout()
plt.show()